# COVID-19 French Maps
Guillaume Rozier, 2020

In [1]:
"""

LICENSE MIT
2020
Guillaume Rozier
Website : http://www.guillaumerozier.fr
Mail : guillaume.rozier@telecomnancy.net

README:s
This file contains script that generate France maps and GIFs. 
Single images are exported to folders in 'charts/image/france'. GIFs are exported to 'charts/image/france'.
I'm currently cleaning this file, please ask me is something is not clear enough!
Requirements: please see the imports below (use pip3 to install them).

"""

"\n\nLICENSE MIT\n2020\nGuillaume Rozier\nWebsite : http://www.guillaumerozier.fr\nMail : guillaume.rozier@telecomnancy.net\n\nREADME:s\nThis file contains script that generate France maps and GIFs. \nSingle images are exported to folders in 'charts/image/france'. GIFs are exported to 'charts/image/france'.\nI'm currently cleaning this file, please ask me is something is not clear enough!\nRequirements: please see the imports below (use pip3 to install them).\n\n"

In [59]:
import france_data_management as data
import pandas as pd
from tqdm import tqdm
import json
import plotly.express as px
from datetime import datetime
import imageio
import multiprocessing
import locale
import shutil
import os
locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')
PATH = "../../"
import subprocess

## Data import

In [3]:
# Import data from Santé publique France
df, df_confirmed, dates, _, _, df_deconf, df_sursaud, df_incid, _ = data.import_data()
df_incid = df_incid[df_incid["cl_age90"] == 0]

  0%|          | 0/8 [00:00<?, ?it/s]/Users/guillaumerozier/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3249: DtypeWarning:

Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.

28it [01:30,  3.86s/it]                      

In [4]:
#df_incid["incidence"] = df_incid["P"]/df_incid["pop"]*100
#df_incid.loc[:,"incidence_color"] = ["white"] * len(df_incid)
for dep in pd.unique(df_incid["dep"].values):
    df_incid.loc[df_incid["dep"] == dep,"incidence"] = df_incid["P"].rolling(window=7).sum()/df_incid["pop"]*100000
df_incid.loc[:,"incidence_color"] = ['Rouge (>50)' if x >= 50 else 'Orange (25-50)' if x >= 25 else 'Vert (<25)' for x in df_incid['incidence']]

In [13]:
"""# Download and import data from INSEE
dict_insee = pd.read_excel('data/france/deces_quotidiens_departement.xlsx', header=[3], index_col=None, sheet_name=None, usecols='A:H', nrows=44)
dict_insee.pop('France')
dict_insee.pop('Documentation')

for key in dict_insee:
    dict_insee[key]["dep"] = [key for i in range(len(dict_insee[key]))]
    
df_insee = pd.concat(dict_insee)
df_insee = df_insee.rename(columns={"Ensemble des communes": "dc20", "Ensemble des communes.1": "dc19", "Ensemble des communes.2": "dc18", "Date d'événement": "jour"})
df_insee = df_insee.drop(columns=['Communes à envoi dématérialisé au 1er avril 2020 (1)', 'Communes à envoi dématérialisé au 1er avril 2020 (1)', 'Communes à envoi dématérialisé au 1er avril 2020 (1)', 'Unnamed: 7'])
df_insee["moy1819"] = (df_insee["dc19"] + df_insee["dc20"])/2
df_insee["surmortalite20"] = (df_insee["dc20"] - df_insee["moy1819"])/df_insee["moy1819"]*100
df_insee['jour'] = pd.to_datetime(df_insee['jour'])
df_insee['jour'] = df_insee['jour'].dt.strftime('%Y-%m-%d')

dates_insee = list(dict.fromkeys(list(df_insee.dropna()['jour'].values))) """

'# Download and import data from INSEE\ndict_insee = pd.read_excel(\'data/france/deces_quotidiens_departement.xlsx\', header=[3], index_col=None, sheet_name=None, usecols=\'A:H\', nrows=44)\ndict_insee.pop(\'France\')\ndict_insee.pop(\'Documentation\')\n\nfor key in dict_insee:\n    dict_insee[key]["dep"] = [key for i in range(len(dict_insee[key]))]\n    \ndf_insee = pd.concat(dict_insee)\ndf_insee = df_insee.rename(columns={"Ensemble des communes": "dc20", "Ensemble des communes.1": "dc19", "Ensemble des communes.2": "dc18", "Date d\'événement": "jour"})\ndf_insee = df_insee.drop(columns=[\'Communes à envoi dématérialisé au 1er avril 2020 (1)\', \'Communes à envoi dématérialisé au 1er avril 2020 (1)\', \'Communes à envoi dématérialisé au 1er avril 2020 (1)\', \'Unnamed: 7\'])\ndf_insee["moy1819"] = (df_insee["dc19"] + df_insee["dc20"])/2\ndf_insee["surmortalite20"] = (df_insee["dc20"] - df_insee["moy1819"])/df_insee["moy1819"]*100\ndf_insee[\'jour\'] = pd.to_datetime(df_insee[\'jour\'

In [14]:
"""df_insee_france = df_insee.groupby('jour').sum().reset_index()
df_insee_france["surmortalite20"] = (df_insee_france["dc20"] - df_insee_france["moy1819"])/df_insee_france["moy1819"]"""

'df_insee_france = df_insee.groupby(\'jour\').sum().reset_index()\ndf_insee_france["surmortalite20"] = (df_insee_france["dc20"] - df_insee_france["moy1819"])/df_insee_france["moy1819"]'

<br>
<br>

## Function definition

In [5]:
with open(PATH+'data/france/dep.geojson') as response:
    depa = json.load(response)

In [133]:
def map_gif(dates, imgs_folder, df, type_ppl, legend_title, min_scale, max_scale, colorscale, subtitle, clean_before=True, clean_after=False):
    try:
        if(clean_before):
            shutil.rmtree(imgs_folder)
            os.mkdir(imgs_folder)
    except:
        print("folder not removed")
    
    i=1
    
    df = df[df['jour'].isin(dates)]
    files = os.listdir(imgs_folder)
    
    for date in tqdm(dates):
        if "{}.jpeg".format(date) in files:
            print("map already generated", (imgs_folder+"/{}.jpeg").format(date))
            continue
        
        if max_scale == -1:
            max_scale = df[type_ppl].max()
        df_map = pd.melt(df, id_vars=['jour','dep'], value_vars=[type_ppl])
        df_map = df_map[df_map["jour"] == date]

        fig = px.choropleth(geojson=depa, 
                            locations=df_map['dep'], 
                            color=df_map['value'],
                            color_continuous_scale = colorscale,
                            range_color=(min_scale, max_scale),
                            featureidkey="properties.code",
                            scope='europe',
                            labels={'color':legend_title}
                                  )
        date_title = datetime.strptime(date, '%Y-%m-%d').strftime('%d %B')
        
        fig.update_geos(fitbounds="locations", visible=False)
        
        var_hab = 'pour 100k. hab.'
        pourcent = ''
        
        val_mean = round(df_map['value'].mean(), 1)
        
        n = len(dates)
        progression = round((i / n) * 50)
        progressbar = progression * '█' + (50 - progression) * '░'
        i += 1
        
        if type_ppl == 'surmortalite20':
            var_hab = ''
            pourcent = " %"
            if val_mean < 0:
                val_mean = "– " + str(abs(val_mean))
            else:
                val_mean = "+ " + str(val_mean)
                
        val_mean = str(val_mean).replace(".", ",")
        
        fig.update_layout(
            margin={"r":0,"t":0,"l":0,"b":0},
            title={
            'text': "{}".format(date_title),
            'y':0.95,
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top'},
            titlefont = dict(
            size=30),
            annotations = [
                dict(
                    x=0.54,
                    y=0.08,
                    xref='paper',
                    yref='paper',
                    xanchor = 'center',
                    text='Source : Santé publique France. Auteur : @guillaumerozier - CovidTracker.fr',
                    showarrow = False
                ),
                dict(
                    x=0.54,
                    y=0.03,
                    xref = 'paper',
                    yref = 'paper',
                    text = "", #progressbar,
                    xanchor = 'center',
                    showarrow = False,
                    font=dict(
                        size=9
                            )
                ),
                dict(
                    x=0.07,
                    y=0.47,
                    xref='paper',
                    yref='paper',
                    xanchor='left',
                    text='Moyenne France',
                    showarrow = False,
                    font=dict(
                        size=14
                            )
                ),
                dict(
                    x=0.07,
                    y=0.50,
                    xref='paper',
                    yref='paper',
                    xanchor='left',
                    text='{}{}'.format(val_mean, pourcent),
                    showarrow = False,
                    font=dict(
                        size=25
                            )
                ),
                
                dict(
                    x=0.07,
                    y=0.45,
                    xref='paper',
                    yref='paper',
                    xanchor='left',
                    text = var_hab,
                    showarrow = False,
                    font=dict(
                        size=14
                            )
                ),
                dict(
                    x=0.55,
                    y=0.9,
                    xref='paper',
                    yref='paper',
                    text=subtitle,
                    showarrow = False,
                    font=dict(
                        size=20
                            )
                )]
             ) 
        
        fig.update_geos(
            #center=dict(lon=-30, lat=-30),
            projection_rotation=dict(lon=12, lat=30, roll=8),
            #lataxis_range=[-50,20], lonaxis_range=[0, 200]
        )
        fig.write_image((imgs_folder+"/{}.jpeg").format(date), scale=2, width=900, height=700)
        
        if date==max(dates):
            fig.write_image((imgs_folder+"/latest.jpeg"), scale=2, width=900, height=700)
            
    if clean_after:
        for file in files:
            if file[:-5] < min(dates):
                os.remove(imgs_folder+"/"+file)
        
    return max_scale

def build_gif(file_gif, imgs_folder, dates):
    print(sorted(dates))
    i=0
    with imageio.get_writer(file_gif, mode='I', duration=0.3) as writer: 
        for idx,date in enumerate(dates):
            image = imageio.imread((imgs_folder+"/{}.jpeg").format(date))
            print("appending", date)
            writer.append_data(image)
            i+=1
            
            if idx==len(dates)-1:
                for _ in range(10):
                    image_last = imageio.imread((imgs_folder+"/{}.jpeg").format(date))
                    writer.append_data(image_last)
                    print("appending (last)", date)
                    
    subprocess.run(["gifsicle", "-i", file_gif, "--optimize=1", "--scale=0.6", "--colors=180", "-o", file_gif[:-4]+"_opti.gif"])
    os.remove(file_gif)

In [7]:
#build_map(df_deconf, img_folder="images/charts/france/deconf_synthese/{}.png", title="Départements déconfinés le 11/05")


In [8]:
def build_map_indic1(data_df, img_folder, legend_title="legend_title", title="title"):
    dates_deconf = list(dict.fromkeys(list(data_df['date_de_passage'].values))) 
    date = dates_deconf[-1]
    
    data_df = data_df[data_df["date_de_passage"] == date]
    
    fig = px.choropleth(geojson = depa, 
                        locations = data_df['dep'], 
                        featureidkey="properties.code",
                        color = data_df['taux_corona'],
                        scope='europe',
                        range_color=(0, 0.1),
                        #labels={'red':"Couleur", 'orange':'bla', 'green':'lol'},
                        #color_discrete_sequence = ["green", "orange", "red"],
                        #color_discrete_map = {"vert":"green", "orange":"orange", "rouge":"red"}
                        #category_orders = {"indic_synthese" :["vert", "orange", "rouge"]}
                              )
    date_title = datetime.strptime(dates_deconf[-1], '%Y-%m-%d').strftime('%d %B')

    fig.update_geos(fitbounds="locations", visible=False)

    fig.update_layout(
        margin={"r":0,"t":20,"l":0,"b":0},
        title={
            'text': title,
            'y':0.98,
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top'},
        
        titlefont = dict(
            size=30),
        
        annotations = [
            dict(
                x=0.54,
                y=0.03,
                xref='paper',
                yref='paper',
                xanchor = 'center',
                text='Source : Ministère de la Santé. Auteur : @guillaumerozier.',
                showarrow = False
            ),

            dict(
                x=0.55,
                y=0.94,
                xref='paper',
                yref='paper',
                text= "Mis à jour le {}".format(date_title),
                showarrow = False,
                font=dict(
                    size=20
                        )
            )]
         ) 

    fig.update_geos(
        #center=dict(lon=-30, lat=-30),
        projection_rotation=dict(lon=12, lat=30, roll=8),
        #lataxis_range=[-50,20], lonaxis_range=[0, 200]
    )
    #fig.show()
    if date == dates_deconf[-1]:
        fig.write_image(img_folder.format("latest"), scale=2, width=1200, height=800)
    fig.write_image(img_folder.format(date), scale=2, width=1200, height=800)
    


<br>

<br>

<br>

<br>

## Function calls

In [134]:
def dep_map():
    # GIF carte nb réanimations par habitant
    imgs_folder = PATH+"images/charts/france/dep-map-img"
    sub = 'Nombre de <b>personnes en réanimation</b> <br>par habitant de chaque département.'
    map_gif(dates[-30:], imgs_folder, df = df, type_ppl = "rea_deppop", legend_title="réan./100k hab", min_scale = 0, max_scale=23, \
            colorscale = [[0, "green"], [0.04, "#ffcc66"], [0.6, "#f50000"], [0.8, "#b30000"], [1, "#3d0000"]], subtitle=sub, clean_before=False, clean_after=False)
    build_gif(file_gif = PATH+"images/charts/france/dep-map.gif", imgs_folder = PATH+"images/charts/france/dep-map-img", dates=dates[-30:])
#dep_map()



100%|██████████| 30/30 [00:00<00:00, 6717.34it/s]


map already generated ../../images/charts/france/dep-map-img/2020-11-12.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-13.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-14.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-15.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-16.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-17.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-18.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-19.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-20.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-21.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-22.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-23.jpeg
map already generated ../../images/charts/france/dep-map-img/2020-11-24.jpeg

In [10]:
def dep_map_dc_cum():
    # GIF carte décès cumulés par habitant
    imgs_folder = PATH+"images/charts/france/dep-map-img-dc-cum"
    sub = 'Nombre de <b>décès cumulés</b> <br>par habitant de chaque département.'
    map_gif(dates[-30:], imgs_folder, df = df, type_ppl = "dc_deppop", legend_title="décès/100k hab", min_scale = 0, max_scale=-1, colorscale ="Reds", subtitle=sub)
    build_gif(file_gif = PATH+"images/charts/france/dep-map-dc-cum.gif", imgs_folder = PATH+"images/charts/france/dep-map-img-dc-cum", dates=dates[-30:])

In [11]:
def dep_map_dc_journ():
    # GIF carte décès quotidiens 
    imgs_folder = PATH+"images/charts/france/dep-map-img-dc-journ"
    sub = 'Nombre de <b>décès quotidien</b> <br>par habitant de chaque département.'
    map_gif(dates[-30:], imgs_folder, df = df, type_ppl = "dc_new_deppop", legend_title="décès/100k hab", min_scale = 0, max_scale=-1, colorscale ="Reds", subtitle=sub)
    build_gif(file_gif = PATH+"images/charts/france/dep-map-dc-journ.gif", imgs_folder = PATH+"images/charts/france/dep-map-img-dc-journ", dates=dates[-30:])

In [135]:
def dep_map_incidence():
    # GIF carte décès quotidiens 
    imgs_folder = PATH+"images/charts/france/dep-map-incid"
    dates_incid = sorted(list(dict.fromkeys(list(df_incid.dropna()['jour'].values))))
    
    sub = '<b>Incidence</b> : nombre de cas hebdomadaires <br>pour 100 000 habitants'
    map_gif(dates_incid[-50:], imgs_folder, df = df_incid, type_ppl = "incidence", legend_title="cas sur 7j/100k hab", min_scale = 0, max_scale=800, \
                                    colorscale = [[0, "green"], [0.08, "#ffcc66"], [0.25, "#f50000"], [0.5, "#b30000"], [1, "#3d0000"]], subtitle=sub, clean_before=False, clean_after=True)
    build_gif(file_gif = PATH+"images/charts/france/dep-map-incid.gif", imgs_folder = PATH+"images/charts/france/dep-map-incid", dates=dates_incid[-50:])

#dep_map_incidence()



  0%|          | 0/50 [00:00<?, ?it/s]

  2%|▏         | 1/50 [00:06<05:19,  6.52s/it]

  4%|▍         | 2/50 [00:13<05:16,  6.60s/it]

  6%|▌         | 3/50 [00:19<05:02,  6.44s/it]

  8%|▊         | 4/50 [00:25<04:56,  6.44s/it]

 10%|█         | 5/50 [00:31<04:44,  6.31s/it]

 12%|█▏        | 6/50 [00:37<04:33,  6.22s/it]

 14%|█▍        | 7/50 [00:43<04:24,  6.15s/it]

 16%|█▌        | 8/50 [00:49<04:16,  6.11s/it]

 18%|█▊        | 9/50 [00:55<04:09,  6.08s/it]

 20%|██        | 10/50 [01:01<04:01,  6.04s/it]

 22%|██▏       | 11/50 [01:08<04:05,  6.28s/it]

 24%|██▍       | 12/50 [01:14<03:55,  6.21s/it]

 26%|██▌       | 13/50 [01:20<03:45,  6.10s/it]

 28%|██▊       | 14/50 [01:26<03:39,  6.11s/it]

 30%|███       | 15/50 [01:32<03:33,  6.09s/it]

 32%|███▏      | 16/50 [01:38<03:26,  6.06s/it]

 34%|███▍      | 17/50 [01:45<03:25,  6.23s/it]

 36%|███▌      | 18/50 [01:51<03:16,  6.14s/it]

 38%|███▊      | 19/50 [01:57<03:09,  6.13s/it]

100%|██████████| 50/50 [02:03<00:00,

map already generated ../../images/charts/france/dep-map-incid/2020-11-09.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-10.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-11.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-12.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-13.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-14.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-15.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-16.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-17.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-18.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-19.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-20.jpeg
map already generated ../../images/charts/france/dep

In [76]:
dep_map_incidence()
dep_map()
#dep_map_dc_cum()
dep_map_dc_journ()



100%|██████████| 30/30 [00:00<00:00, 698.05it/s]


  0%|          | 0/30 [00:00<?, ?it/s]

map already generated ../../images/charts/france/dep-map-incid/2020-11-09.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-10.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-11.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-12.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-13.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-14.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-15.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-16.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-17.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-18.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-19.jpeg
map already generated ../../images/charts/france/dep-map-incid/2020-11-20.jpeg
map already generated ../../images/charts/france/dep



  3%|▎         | 1/30 [00:00<00:24,  1.20it/s]

../../images/charts/france/dep-map-incid/2020-11-10.jpeg




  7%|▋         | 2/30 [00:01<00:22,  1.24it/s]

...



 93%|█████████▎| 28/30 [00:13<00:00,  2.21it/s]

../../images/charts/france/dep-map-img-dc-journ/2020-12-10.jpeg




 97%|█████████▋| 29/30 [00:13<00:00,  2.20it/s]

../../images/charts/france/dep-map-img-dc-journ/2020-12-11.jpeg




100%|██████████| 30/30 [00:16<00:00,  1.77it/s]


In [20]:
"""
# INSEE
# GIF mortalité par rapport à 2018 et 2019
imgs_folder = "images/charts/france/dep-map-surmortalite-img/{}.png"
ppl = "surmortalite20"
sub = 'Comparaison de la <b>mortalité journalière</b> entre 2020 <br>et les deux années précédentes.'
map_gif(dates_insee, imgs_folder, df = df_insee.dropna(), type_ppl = ppl, legend_title="Sur-mortalité (%)", min_scale=-50, max_scale=50, colorscale = ["green", "white", "red"], subtitle = sub)
build_gif(file_gif = "images/charts/france/dep-map-surmortalite.gif", imgs_folder = imgs_folder, dates=dates_insee)"""

'\n# INSEE\n# GIF mortalité par rapport à 2018 et 2019\nimgs_folder = "images/charts/france/dep-map-surmortalite-img/{}.png"\nppl = "surmortalite20"\nsub = \'Comparaison de la <b>mortalité journalière</b> entre 2020 <br>et les deux années précédentes.\'\nmap_gif(dates_insee, imgs_folder, df = df_insee.dropna(), type_ppl = ppl, legend_title="Sur-mortalité (%)", min_scale=-50, max_scale=50, colorscale = ["green", "white", "red"], subtitle = sub)\nbuild_gif(file_gif = "images/charts/france/dep-map-surmortalite.gif", imgs_folder = imgs_folder, dates=dates_insee)'

In [21]:
"""# Line chart évolution de la mortalité

import plotly.graph_objects as go
import plotly
fig = go.Figure()

fig.add_trace(go.Scatter(
    x = df_insee_france["jour"],
    y = df_insee_france["surmortalite20"],
    name = "Bilan autre hosp",
    marker_color='black',
    mode="lines+markers",
    opacity=1
))


# Here we modify the tickangle of the xaxis, resulting in rotated labels.
fig.update_layout(
    legend_orientation="v",
    barmode='relative',
    title={
                'text': "Variation de la <b>mortalité en mars 2020</b> par rapport à 2018 et 2019",
                'y':0.95,
                'x':0.5,
                'xanchor': 'center',
                'yanchor': 'top'},
                titlefont = dict(
                size=20),
    xaxis=dict(
        title='',
        tickformat='%d/%m'),
    yaxis_title="Surmortalité (%)",
    
    annotations = [
                dict(
                    x=0,
                    y=1.05,
                    xref='paper',
                    yref='paper',
                    text='Date : {}. Source : INSEE et CSSE. Auteur : @guillaumerozier (Twitter).'.format(datetime.strptime(dates[-1], '%Y-%m-%d').strftime('%d %B %Y')),                    showarrow = False
                )]
                 )

fig.update_layout(
    yaxis = go.layout.YAxis(
        tickformat = '%'
    ),
    annotations = [
                dict(
                    x=0.5,
                    y=1.05,
                    xref='paper',
                    yref='paper',
                    xanchor='center',
                    text='',
                    showarrow = False
                )]
                 )

name_fig = "insee_surmortalite"
fig.write_image("images/charts/france/{}.png".format(name_fig), scale=2, width=1200, height=800)
plotly.offline.plot(fig, filename = 'images/html_exports/france/{}.html'.format(name_fig), auto_open=False)
print("> " + name_fig)

fig.show()"""

'# Line chart évolution de la mortalité\n\nimport plotly.graph_objects as go\nimport plotly\nfig = go.Figure()\n\nfig.add_trace(go.Scatter(\n    x = df_insee_france["jour"],\n    y = df_insee_france["surmortalite20"],\n    name = "Bilan autre hosp",\n    marker_color=\'black\',\n    mode="lines+markers",\n    opacity=1\n))\n\n\n# Here we modify the tickangle of the xaxis, resulting in rotated labels.\nfig.update_layout(\n    legend_orientation="v",\n    barmode=\'relative\',\n    title={\n                \'text\': "Variation de la <b>mortalité en mars 2020</b> par rapport à 2018 et 2019",\n                \'y\':0.95,\n                \'x\':0.5,\n                \'xanchor\': \'center\',\n                \'yanchor\': \'top\'},\n                titlefont = dict(\n                size=20),\n    xaxis=dict(\n        title=\'\',\n        tickformat=\'%d/%m\'),\n    yaxis_title="Surmortalité (%)",\n    \n    annotations = [\n                dict(\n                    x=0,\n                   